# Numerikus módszerek 1. – 8. gyakorlat
## Iterációs módszerek: Jacobi, Gauss–Seidel, relaxált GS, Richardson

**ELTE IK – Krebsz Anna**

---

A feladatok az előadás anyagára épülnek (ea_07, ea_08). Ahol nem jelezzük másképp, próbáld meg **előbb kézzel (papír-ceruza)** elvégezni a számításokat, majd ellenőrizd Pythonnal!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=4, suppress=True)

# Segédfüggvények
def print_matrix(A, name='A'):
    print(f'{name} =')
    for row in A:
        print(' ', row)

def print_vector(v, name='x'):
    print(f'{name} = {v}')

def spectral_radius(B):
    """B mátrix spektrálsugara: max|lambda_i(B)|."""
    return np.max(np.abs(np.linalg.eigvals(B)))

def split_LDU(A):
    """A = L + D + U felbontás."""
    return np.tril(A, -1), np.diag(np.diag(A)), np.triu(A, 1)

---
## 1. feladat – Progonka (Thomas-) módszer

### Emlékeztető

Tridiagonális $A = \mathrm{tridiag}(\beta_{i-1}, \alpha_i, \gamma_i)$ esetén az algoritmus:

**1. lépés (előre):**
$$f_1 = -\frac{\gamma_1}{\alpha_1},\quad g_1 = \frac{b_1}{\alpha_1}, \qquad f_i = -\frac{\gamma_i}{\alpha_i + \beta_{i-1}f_{i-1}}, \quad g_i = \frac{b_i - \beta_{i-1}g_{i-1}}{\alpha_i + \beta_{i-1}f_{i-1}}$$

**2. lépés (vissza):**
$x_n = g_n$, majd $x_i = f_i x_{i+1} + g_i$ ($i = n-1, \ldots, 1$)

---
### 1a. Kézzel: 4×4 rendszer

Oldd meg kézzel a következő rendszert (töltsd ki az üres cellát):

$$\begin{bmatrix} 2 & -1 & 0 & 0 \\ -1 & 2 & -1 & 0 \\ 0 & -1 & 2 & -1 \\ 0 & 0 & -1 & 2 \end{bmatrix} \begin{bmatrix}x_1\\x_2\\x_3\\x_4\end{bmatrix} = \begin{bmatrix}1\\0\\0\\1\end{bmatrix}$$

**Kézzel számított megoldás (töltsd ki!):**

1. lépés:
- $f_1 = $ ..., $g_1 = $ ...
- $f_2 = $ ..., $g_2 = $ ...
- $f_3 = $ ..., $g_3 = $ ...
- $g_4 = $ ...

2. lépés:
- $x_4 = $ ..., $x_3 = $ ..., $x_2 = $ ..., $x_1 = $ ...

### 1b. Progonka implementálása

In [ ]:
def progonka(alpha, beta, gamma, b):
    """
    Tridiagonális LER megoldása progonka módszerrel.
    
    alpha : (n,)   főátló
    beta  : (n-1,) alsó szomszédátló (beta[i] = A[i+1,i])
    gamma : (n-1,) felső szomszédátló (gamma[i] = A[i,i+1])
    b     : (n,)   jobb oldal
    """
    n = len(alpha)
    f = np.zeros(n)
    g = np.zeros(n)

    # TODO: 1. lépés (előre) implementálása
    # ...

    # TODO: 2. lépés (visszafelé) implementálása
    x = np.zeros(n)
    # ...

    return x

### 1c. Ellenőrzés

In [ ]:
alpha = np.array([2.0, 2.0, 2.0, 2.0])
beta  = np.array([-1.0, -1.0, -1.0])
gamma = np.array([-1.0, -1.0, -1.0])
b     = np.array([1.0, 0.0, 0.0, 1.0])

x_prog = progonka(alpha, beta, gamma, b)
print('Progonka:         ', x_prog)

A_full = np.diag(alpha) + np.diag(beta, -1) + np.diag(gamma, 1)
x_ref  = np.linalg.solve(A_full, b)
print('np.linalg.solve:  ', x_ref)
print('Eltérés (max):    ', np.max(np.abs(x_prog - x_ref)))

### 1d. *(Nehezebb)* Köbös spline alkalmazás

Köbös spline interpoláció esetén az ismeretlen $M_i = S_i''(x_i)$ értékekre a következő tridiagonális rendszer adódik ($h_i = x_{i+1} - x_i$ egyenletes rácsra, $h_i = h$):

$$\frac{h}{6} M_{i-1} + \frac{2h}{3} M_i + \frac{h}{6} M_{i+1} = \frac{y_{i+1} - y_i}{h} - \frac{y_i - y_{i-1}}{h}, \quad i = 1, \ldots, n-1$$

természetes spline feltételekkel $M_0 = M_n = 0$.

Oldd meg a rendszert progonkával az alábbi adatokra, majd rajzold fel az interpolált görbét!

$$x_i = 0, 1, 2, 3, 4, \quad y_i = 0, 1, 0, 1, 0$$

In [ ]:
xi = np.array([0, 1, 2, 3, 4], dtype=float)
yi = np.array([0, 1, 0, 1, 0], dtype=float)
n_pts = len(xi)
h = xi[1] - xi[0]  # egyenletes rács

# TODO: Tridiagonális rendszer felírása és megoldása progonkával
# Az M értékek mérete: n_pts - 2 (belső pontok, M_0 = M_n = 0)

# alpha_sp = ...
# beta_sp  = ...
# gamma_sp = ...
# b_sp     = ...
# M_inner  = progonka(alpha_sp, beta_sp, gamma_sp, b_sp)
# M = np.concatenate([[0], M_inner, [0]])

# TODO: Rajz
# ...

---
## 2. feladat – Jacobi-iteráció

### Emlékeztető

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(-\sum_{j \ne i} a_{ij} x_j^{(k)} + b_i\right)$$

### 2a. Kézzel: 2 lépés Jacobi

Végezz el 2 Jacobi-lépést $x^{(0)} = (0,0,0)^\top$ kezdőpontból:

$$A = \begin{bmatrix}4 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 4\end{bmatrix}, \quad b = \begin{bmatrix}3\\2\\3\end{bmatrix}$$

**Kézzel számítva (töltsd ki!):**

$x^{(1)} = $ (..., ..., ...)

$x^{(2)} = $ (..., ..., ...)

Pontos megoldás: $x^* = (1, 1, 1)^\top$. Melyik norma szerint közelít gyorsabban?

### 2b. Jacobi-iteráció implementálása

In [ ]:
def jacobi(A, b, x0, tol=1e-8, max_iter=1000):
    """
    Jacobi-iteráció.
    Visszaad: (x, res_history) – reziduum ∞-normáinak listája iterációnként.
    """
    x = x0.copy().astype(float)
    r = b - A @ x
    d = np.diag(A)
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for _ in range(max_iter):
        # TODO: s = D^{-1} r, majd x := x+s, r := r - A*s
        # s = ...
        # x = ...
        # r = ...
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

### 2c. Konvergencia vizsgálat

In [ ]:
A_J = np.array([[4., -1., 0.],
                [-1., 4., -1.],
                [0., -1., 4.]])
b_J = np.array([3., 2., 3.])
x0  = np.zeros(3)

x_sol, res_hist = jacobi(A_J, b_J, x0, tol=1e-10)
print(f'Megoldás: {x_sol}  ({len(res_hist)-1} iteráció)')

# Log-skálás grafikon
plt.figure(figsize=(7, 4))
plt.semilogy(res_hist, 'b-o', markersize=4)
plt.xlabel('Iteráció')
plt.ylabel('‖r^(k)‖∞')
plt.title('Jacobi-iteráció konvergenciája')
plt.grid(True)
plt.tight_layout()
plt.show()

# Spektrálsugár
L, D, U = split_LDU(A_J)
B_J = -np.linalg.inv(D) @ (L + U)
rho_J = spectral_radius(B_J)
print(f'ρ(B_J) = {rho_J:.4f}  => konvergens: {rho_J < 1}')

### 2d. Divergencia demonstrálása

Vizsgáld a következő **nem** s.d.d. mátrixra a Jacobi-iterációt!

In [ ]:
# Nem s.d.d. mátrix: pl. a sorok összege meghaladja a diagonális elemet
A_div = np.array([[1., 2., 0.],
                  [2., 1., 2.],
                  [0., 2., 1.]])
b_div = np.array([1., 0., 1.])

# Spektrálsugár
L_d, D_d, U_d = split_LDU(A_div)
B_div = -np.linalg.inv(D_d) @ (L_d + U_d)
rho_div = spectral_radius(B_div)
print(f'ρ(B_J) = {rho_div:.4f}  (> 1: divergens várható)')

# Jacobi futtatása (korlátozott lépés)
x_d, res_d = jacobi(A_div, b_div, np.zeros(3), tol=1e-8, max_iter=30)

plt.figure(figsize=(7, 4))
plt.plot(res_d, 'r-o', markersize=4)
plt.xlabel('Iteráció')
plt.ylabel('‖r^(k)‖∞')
plt.title('Divergens Jacobi-iteráció')
plt.grid(True)
plt.tight_layout()
plt.show()

---
## 3. feladat – Gauss–Seidel vs. Jacobi

### 3a. Gauss–Seidel implementálása

In [ ]:
def gauss_seidel(A, b, x0, tol=1e-8, max_iter=1000):
    """
    Gauss–Seidel iteráció.
    (D+L) s^(k) = r^(k)  LER megoldása előretolással.
    """
    x = x0.copy().astype(float)
    r = b - A @ x
    n = len(b)
    L, D, U = split_LDU(A)
    LD = L + D
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for _ in range(max_iter):
        # TODO: Oldd meg (D+L)s = r előretolással
        s = np.zeros(n)
        for i in range(n):
            pass  # s[i] = ...

        x = x + s
        r = r - A @ s
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

### 3b. Összehasonlítás tridiagonális rendszeren

In [ ]:
# Ugyanaz a mátrix mint az előző feladatban
x_GS, res_GS = gauss_seidel(A_J, b_J, x0, tol=1e-10)
x_JA, res_JA = jacobi(A_J, b_J, x0, tol=1e-10)

print(f'Jacobi:       {len(res_JA)-1} iteráció')
print(f'Gauss–Seidel: {len(res_GS)-1} iteráció')

plt.figure(figsize=(8, 5))
plt.semilogy(res_JA, 'b-o', markersize=3, label=f'Jacobi ({len(res_JA)-1} it.)')
plt.semilogy(res_GS, 'r-s', markersize=3, label=f'Gauss–Seidel ({len(res_GS)-1} it.)')
plt.xlabel('Iteráció')
plt.ylabel('‖r^(k)‖∞')
plt.title('Jacobi vs. Gauss–Seidel')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### 3c. Spektrálsugarak összehasonlítása

Ellenőrizd numerikusan, hogy tridiagonális mátrix esetén $\varrho(B_S) = \varrho(B_J)^2$ teljesül-e!

In [ ]:
# TODO: Számítsd ki ρ(B_J) és ρ(B_S) értékeket, és hasonlítsd össze ρ(B_J)²-tel
L, D, U = split_LDU(A_J)

# B_J = ...
# B_S = ...
# rho_J = spectral_radius(B_J)
# rho_S = spectral_radius(B_S)
# print(f'ρ(B_J) = {rho_J:.6f}')
# print(f'ρ(B_S) = {rho_S:.6f}')
# print(f'ρ(B_J)² = {rho_J**2:.6f}')
# print(f'Egyenlők (numerikusan): {np.isclose(rho_S, rho_J**2)}')

---
## 4. feladat – Relaxált Gauss–Seidel $S(\omega)$

### 4a. $S(\omega)$ implementálása

In [ ]:
def relaxed_gauss_seidel(A, b, x0, omega=1.0, tol=1e-8, max_iter=2000):
    """
    Relaxált Gauss–Seidel S(omega) iteráció.
    (D + omega*L) s^(k) = omega * r^(k)
    """
    x = x0.copy().astype(float)
    r = b - A @ x
    n = len(b)
    L, D, _ = split_LDU(A)
    DwL = D + omega * L
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for _ in range(max_iter):
        # TODO: Oldd meg (D + omega*L) s = omega*r előretolással
        rhs = omega * r
        s = np.zeros(n)
        for i in range(n):
            pass  # s[i] = ...

        x = x + s
        r = r - A @ s
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

### 4b. $\omega$-sweep: iterációszám az $\omega$ paraméter függvényében

In [ ]:
omega_vals = np.linspace(0.1, 1.95, 80)
iter_counts = []

for om in omega_vals:
    _, res = relaxed_gauss_seidel(A_J, b_J, x0, omega=om, tol=1e-8, max_iter=500)
    iter_counts.append(len(res) - 1)

# TODO: Rajzold az iterációszámot omega függvényében!
# plt.figure(...)
# plt.plot(...)
# ...
# plt.show()

### 4c. Elméleti optimális $\omega_0$

Szimmetrikus, pozitív definit, tridiagonális esetben:
$$\omega_0 = \frac{2}{1 + \sqrt{1 - \varrho(B_J)^2}}$$

In [ ]:
# TODO: Számítsd ki az elméleti optimális omega_0-t és rajzold be a grafikonba!
# rho_J = spectral_radius(B_J_matrix)  # ahol B_J_matrix a Jacobi átmenet mátrix
# omega_opt = ...
# print(f'Elméleti ω₀ = {omega_opt:.4f}')

# Ellenőrzés: hányadik omega_vals értékhez van a legközelebb?
# idx_min = np.argmin(iter_counts)
# print(f'Numerikusan legjobb ω ≈ {omega_vals[idx_min]:.3f} ({iter_counts[idx_min]} iteráció)')

### 4d. Elméleti kérdés

**Miért nem konvergál $S(\omega)$, ha $\omega \ge 2$?**

*(Töltsd ki az alábbi markdown cellát a válasszal!)*

**Válasz:** ...

---
## 5. feladat – Richardson-iteráció

### Emlékeztető

$$x^{(k+1)} = (I - pA)x^{(k)} + pb, \qquad s^{(k)} = p\,r^{(k)}$$

Szimm.+pozdef. ($\lambda_1 = m \le \cdots \le \lambda_n = M$):
$$p_0 = \frac{2}{M + m}, \qquad q = \frac{M - m}{M + m}$$

### 5a. Richardson implementálása

In [ ]:
def richardson(A, b, x0, p, tol=1e-8, max_iter=1000):
    """Richardson-iteráció R(p) paraméterrel."""
    x = x0.copy().astype(float)
    r = b - A @ x
    res_history = [np.linalg.norm(r, ord=np.inf)]

    for _ in range(max_iter):
        # TODO: s = p*r, x := x+s, r := r - A*s
        # s = ...
        # x = ...
        # r = ...
        res_history.append(np.linalg.norm(r, ord=np.inf))
        if res_history[-1] < tol:
            break

    return x, res_history

### 5b. Optimális paraméter kiszámítása (kézzel is!)

Adott:
$$A = \begin{bmatrix}3 & -1 \\ -1 & 3\end{bmatrix}, \quad b = \begin{bmatrix}1 \\ 5\end{bmatrix}$$
A mátrix sajátértékei: $\lambda_1 = 2$, $\lambda_2 = 4$.

**Kézzel (töltsd ki!):**

- $m = $ ..., $M = $ ...
- Konvergens $p$ tartomány: $(0, $ ... $)$
- Optimális $p_0 = $ ...
- Kontrakciós együttható $q = $ ...

In [ ]:
A_R = np.array([[3., -1.], [-1., 3.]])
b_R = np.array([1., 5.])
x0_R = np.zeros(2)

# TODO: Számítsd ki a sajátértékeket, p0-t és q-t numerikusan
# eigvals = ...
# m_R, M_R = ...
# p0 = ...
# q  = ...
# print(f'p₀ = {p0:.4f},  q = {q:.4f}')

### 5c. Különböző $p$ értékek összehasonlítása

In [ ]:
# TODO: Futtasd p = 0.1, 0.2, p0, 0.4, 0.49 értékekre és rajzold a konvergenciát!
# p_vals = [0.1, 0.2, p0, 0.4, 0.49]
# ...
# plt.figure(...)
# for pv in p_vals:
#     ...
# plt.show()

### 5d. Richardson $\leftrightarrow$ Csillapított Jacobi kapcsolat

**Állítás:** Ha $D = \mathrm{diag}(a_{11}, \ldots, a_{nn})$ és a Richardson-iterációt a $D^{-1}Ax = D^{-1}b$ LER-re alkalmazzuk $p = 1$ esetén, az eredeti LER-re felírt $J(p)$ csillapított Jacobi-iterációt kapjuk.

Ellenőrizd ezt numerikusan az $A_J$ mátrixon! ($A_J$ diagonálisan domináns, $D = 4I$.)

In [ ]:
# TODO: 
# 1. Képezd D^{-1}A és D^{-1}b értékeket
# 2. Futtass Richardson-iterációt p=1-gyel a transzformált rendszerre
# 3. Hasonlítsd össze a J(1/4) csillapított Jacobi-iterációval az eredeti rendszeren
# (A J csillapított Jacobi omega=1/4-es esetnek felel meg a J(omega) def. alapján?)
# ...

# Segítség: a J(omega) lépése: s = omega * r / d, ahol d = diag(A)
# A Richardson lépése: s = p * r_transzf, ahol r_transzf = D^{-1} * r_eredeti
# => s = p * D^{-1} * r = (p/d_ii) * r_i komponensenként

---
## 6. feladat – Önálló összehasonlítás nagy rendszeren

Tekintsd az $n$-dimenziós **Laplace-diszkretizáció** tridiagonális rendszerét:

$$A = \mathrm{tridiag}(-1, 2, -1), \quad b_i = h^2 f(x_i), \quad h = \frac{1}{n+1}, \quad f(x) = \sin(\pi x)$$

Hasonlítsd össze az összes módszert $n = 50$ és $n = 100$ esetén!

In [ ]:
def laplace_system(n):
    """Laplace-diszkretizáció tridiagonális rendszere."""
    h = 1.0 / (n + 1)
    x = np.linspace(h, 1 - h, n)
    alpha = np.full(n, 2.0)
    beta  = np.full(n - 1, -1.0)
    gamma = np.full(n - 1, -1.0)
    b = h**2 * np.sin(np.pi * x)
    A = np.diag(alpha) + np.diag(beta, -1) + np.diag(gamma, 1)
    return A, alpha, beta, gamma, b


for n in [50, 100]:
    A, alpha, beta, gamma, b = laplace_system(n)
    x0 = np.zeros(n)
    tol = 1e-8

    print(f'\n=== n = {n} ===')

    # Jacobi
    t0 = time.time()
    _, res_J = jacobi(A, b, x0, tol=tol, max_iter=5000)
    t_J = time.time() - t0
    print(f'Jacobi:          {len(res_J)-1:4d} it., reziduum: {res_J[-1]:.2e}, idő: {t_J:.3f}s')

    # Gauss-Seidel
    t0 = time.time()
    _, res_GS = gauss_seidel(A, b, x0, tol=tol, max_iter=5000)
    t_GS = time.time() - t0
    print(f'Gauss–Seidel:    {len(res_GS)-1:4d} it., reziduum: {res_GS[-1]:.2e}, idő: {t_GS:.3f}s')

    # Jacobi B_J spektrálsugara (optimális omega és p0 számításához)
    L_m, D_m, U_m = split_LDU(A)
    B_J_m = -np.linalg.inv(D_m) @ (L_m + U_m)
    rho_BJ = spectral_radius(B_J_m)
    omega_opt = 2 / (1 + np.sqrt(1 - rho_BJ**2))

    # Relaxált GS
    t0 = time.time()
    _, res_RGS = relaxed_gauss_seidel(A, b, x0, omega=omega_opt, tol=tol, max_iter=5000)
    t_RGS = time.time() - t0
    print(f'Relaxált GS (ω₀={omega_opt:.3f}): {len(res_RGS)-1:4d} it., reziduum: {res_RGS[-1]:.2e}, idő: {t_RGS:.3f}s')

    # Richardson
    eigvals = np.linalg.eigvalsh(A)
    m_R, M_R = eigvals.min(), eigvals.max()
    p0 = 2 / (M_R + m_R)
    t0 = time.time()
    _, res_Rich = richardson(A, b, x0, p=p0, tol=tol, max_iter=5000)
    t_Rich = time.time() - t0
    print(f'Richardson (p₀={p0:.4f}): {len(res_Rich)-1:4d} it., reziduum: {res_Rich[-1]:.2e}, idő: {t_Rich:.3f}s')

    # Progonka referencia
    t0 = time.time()
    h = 1.0 / (n + 1)
    x_prog = progonka(np.full(n, 2.0), np.full(n - 1, -1.0), np.full(n - 1, -1.0), b)
    t_prog = time.time() - t0
    res_prog = np.linalg.norm(b - A @ x_prog, ord=np.inf)
    print(f'Progonka:        direkt, reziduum: {res_prog:.2e}, idő: {t_prog:.3f}s')

In [ ]:
# Összehasonlítás vizualizációja (n=100)
n = 100
A, alpha, beta, gamma, b = laplace_system(n)
x0 = np.zeros(n)

L_m, D_m, U_m = split_LDU(A)
B_J_m = -np.linalg.inv(D_m) @ (L_m + U_m)
rho_BJ = spectral_radius(B_J_m)
omega_opt = 2 / (1 + np.sqrt(1 - rho_BJ**2))
eigvals = np.linalg.eigvalsh(A)
p0 = 2 / (eigvals.max() + eigvals.min())

_, res_J100   = jacobi(A, b, x0, tol=1e-10, max_iter=5000)
_, res_GS100  = gauss_seidel(A, b, x0, tol=1e-10, max_iter=5000)
_, res_RGS100 = relaxed_gauss_seidel(A, b, x0, omega=omega_opt, tol=1e-10, max_iter=5000)
_, res_R100   = richardson(A, b, x0, p=p0, tol=1e-10, max_iter=5000)

plt.figure(figsize=(10, 6))
plt.semilogy(res_J100,   'b-',  linewidth=1.5, label=f'Jacobi ({len(res_J100)-1} it.)')
plt.semilogy(res_GS100,  'g-',  linewidth=1.5, label=f'Gauss–Seidel ({len(res_GS100)-1} it.)')
plt.semilogy(res_RGS100, 'r-',  linewidth=2,   label=f'Relaxált GS ω₀={omega_opt:.3f} ({len(res_RGS100)-1} it.)')
plt.semilogy(res_R100,   'm--', linewidth=1.5, label=f'Richardson p₀={p0:.4f} ({len(res_R100)-1} it.)')
plt.xlabel('Iteráció')
plt.ylabel('‖r^(k)‖∞')
plt.title(f'Iterációs módszerek összehasonlítása (n={n})')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Elemzési kérdések (töltsd ki!):**

1. Melyik módszer konvergál a leggyorsabban? Megfelel-e az elméleti várakozásoknak?
2. Hogyan változik az iterációszám, ha $n$-et megduplázzuk?
3. Miért van nagy különbség a progonka és az iterációs módszerek futási ideje között?

**Válaszok:** ...